<a href="https://colab.research.google.com/github/MichalPietruszevski/phd/blob/main/wykluczenia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import openpyxl

# Google colab

In [3]:
from google.colab import drive
drive.mount('/content/drive')
DANE = "/content/drive/MyDrive/DOKTORAT/ANALIZY/DANE/baza_danych_270426.xlsx"
df = pd.read_excel(DANE)

Mounted at /content/drive


# Local

In [5]:
DANE = "../DANE/baza_danych_270426.xlsx"
df = pd.read_excel(DANE)

FileNotFoundError: [Errno 2] No such file or directory: '../DANE/baza_danych_270426.xlsx'

Dobra jakich potrzebujemy kolumn do wykluczen  
Potrzebuje:  
ank_zdro_palenie_teraz   
  
ank_zdro_alko_U  



Do kontroli:  
ank_zwi_tn  
ank_zwi_status
ank_zdro_leki_U  
ank_zdro_chor_przew_tn  

In [6]:
# Merge categories for ank_zdro_alko
def categorize_alko(val):
    if pd.isna(val):
        return val
    val_str = str(val).strip().lower()

    # 4 razy w tygodniu i częściej
    if '4 ra' in val_str and 'tygodni' in val_str:
        return '4 razy w tygodniu i częściej'

    # 2-3 razy w tygodniu
    if '2-3 razy w tygodniu' in val_str:
        return '2-3 razy w tygodniu'

    # Raz w miesiącu lub rzadziej (includes "2-4 razy w miesiącu")
    if 'miesiąc' in val_str or 'rzadziej' in val_str:
        return 'raz w miesiącu lub rzadziej'

    # Nigdy
    if 'nigdy' in val_str:
        return 'nigdy'

    # Default for numeric or unclear values
    return 'raz w miesiącu lub rzadziej'

df['ank_zdro_alko_U'] = df['ank_zdro_alko'].apply(categorize_alko)
df.ank_zdro_alko_U.value_counts()

,count
ank_zdro_alko_U,
raz w miesiącu lub rzadziej,137
2-3 razy w tygodniu,36
nigdy,31
4 razy w tygodniu i częściej,6


In [7]:
# Create the ank_zdro_leki_U column with TAK/NIE labels
def categorize_leki(val):
    if pd.isna(val) or str(val).strip() in ['BRAK', '', 'nan']:
        return 'NIE'
    else:
        return 'TAK'

df['ank_zdro_leki_U'] = df['ank_zdro_leki'].apply(categorize_leki)

In [8]:
df.ank_zdro_leki_U.value_counts()

,count
ank_zdro_leki_U,
NIE,134
TAK,77


In [9]:
df.ank_zwi_tn.value_counts()

,count
ank_zwi_tn,
TAK,185
NIE,24


# Cukrzyca

In [ ]:
for i in df.columns.tolist(): print(i) if i.startswith("ank_zdro") else None

In [21]:
# 1. Rozszerzona funkcja czyszcząca
def clean_choroby_v2(val):
    if pd.isna(val) or str(val).lower().strip() in ['brak', 'nie', 'nie dotyczy', '-', 'nan']:
        return 'Brak'

    s = str(val).lower().strip()

    # Mapowanie i poprawa literówek
    if 'cukrzyca' in s:
        return 'Cukrzyca'

    return s.capitalize()

# 2. Tworzenie nowej zmiennej
df['ank_zdro_chor_przew_cukrz'] = df['ank_zdro_chor_przew_tx'].apply(clean_choroby_v2)

# 3. Sprawdzenie wyników po poprawkach
print("Zaktualizowane kategorie (ank_zdro_chor_przew_U):")
print(df['ank_zdro_chor_przew_cukrz'].value_counts())

Zaktualizowane kategorie (ank_zdro_chor_przew_U):
ank_zdro_chor_przew_cukrz
Brak                                                                                      138
Nadciśnienie                                                                               10
Cukrzyca                                                                                    5
Nadciśnienie od 2 lat                                                                       2
Nadciśnienie 3 lata                                                                         2
Nadciśnienie od 13 lat                                                                      2
Choroba scheuermanna                                                                        1
Nadciśnienie od 20 lat                                                                      1
Niedoczynność tarczycy od 8 lat                                                             1
Nadciśnienie (od 10 lat)                                                      

TODO:  
[] ogarnąć choroby uprościć  
[] ogarnać leki na cukrzycę  
[] ogarnąć hemoglobinę glikowaną

'par_hg_glikowana_mol' - poniżej 5,7% (39 mmol/mol).
'par_ob' - U mężczyzn w wieku 15–50 lat OB powinno być mniejsze niż 15 mm/h, a powyżej 50 roku życia ten zakres wzrasta, niemniej nie powinien być wyższy niż 20 mm/h


In [29]:
import pandas as pd

# Konwersja kolumn na format numeryczny
df['par_hg_glikowana_mol'] = pd.to_numeric(df['par_hg_glikowana_mol'], errors='coerce')
df['par_ob'] = pd.to_numeric(df['par_ob'], errors='coerce')

# 1. Hemoglobina glikowana - norma poniżej 39 mmol/mol
def check_hg_norm(val):
    if pd.isna(val): return None
    return 1 if val < 39 else 0

df['par_hg_glikowana_norma'] = df['par_hg_glikowana_mol'].apply(check_hg_norm)

# 2. OB (Odczyn Biernackiego) - traktujemy wszystkich jako > 50 lat
def check_ob_norm_v2(val):
    if pd.isna(val): return None
    # Norma dla osób > 50 roku życia: < 20 mm/h
    return 1 if val < 20 else 0

df['par_ob_norma'] = df['par_ob'].apply(check_ob_norm_v2)

# Wyświetlenie podsumowania
print("Podsumowanie norm dla Hemoglobiny Glikowanej (norma < 39):")
print(df['par_hg_glikowana_norma'].value_counts(dropna=False))
print("\nPodsumowanie norm dla OB (wszyscy jako >50 lat, norma < 20):")
print(df['par_ob_norma'].value_counts(dropna=False))

display(df[['par_hg_glikowana_mol', 'par_hg_glikowana_norma', 'par_ob', 'par_ob_norma']].head(10))

Podsumowanie norm dla Hemoglobiny Glikowanej (norma < 39):
par_hg_glikowana_norma
1.0    161
0.0     49
NaN      1
Name: count, dtype: int64

Podsumowanie norm dla OB (wszyscy jako >50 lat, norma < 20):
par_ob_norma
1.0    179
0.0     31
NaN      1
Name: count, dtype: int64


,par_hg_glikowana_mol,par_hg_glikowana_norma,par_ob,par_ob_norma
0,35.1,1.0,5.0,1.0
1,36.7,1.0,6.0,1.0
2,35.4,1.0,3.0,1.0
3,38.0,1.0,5.0,1.0
4,32.9,1.0,2.0,1.0
5,34.0,1.0,2.0,1.0
6,35.4,1.0,5.0,1.0
7,34.1,1.0,4.0,1.0
8,42.9,0.0,23.0,0.0
9,37.2,1.0,4.0,1.0


In [25]:
import re

# Zaktualizowana lista słów kluczowych o Galvus i fragmenty łapiące literówki
diabetes_keywords = ['metform', 'glucophag', 'siofor', 'siofou', 'insulina', 'jardiance', 'forxiga', 'victoza', 'ozempic', 'rybelsus', 'amaryl', 'diaprel', 'gliclada', 'galvus']

def check_diabetes_meds(val):
    if pd.isna(val):
        return 0
    s = str(val).lower()
    if any(keyword in s for keyword in diabetes_keywords):
        return 1
    return 0

# Tworzenie/Aktualizacja kolumny
df['ank_zdro_leki_cukrzyca_D'] = df['ank_zdro_leki'].apply(check_diabetes_meds)

# Wyświetlenie osób po aktualizacji
print("Osoby zidentyfikowane jako przyjmujące leki na cukrzycę:")
display(df[df['ank_zdro_leki_cukrzyca_D'] == 1][['ank_zdro_leki', 'ank_zdro_leki_cukrzyca_D']])

print("\nPodsumowanie nowej kolumny dummy:")
print(df['ank_zdro_leki_cukrzyca_D'].value_counts())

Osoby zidentyfikowane jako przyjmujące leki na cukrzycę:


,ank_zdro_leki,ank_zdro_leki_cukrzyca_D
57,Lorista H 1/2x50 mg; Betaloc 20K 1/2x 95mg; Fo...,1
72,Comlessa 8+10+250; Nobilet; SioFou 850,1
112,"Glucophag, Prestarium 5mg. Romazic, Milurit 100mg",1
121,"Medal, tolura, milurit, romazic, ozempic",1
180,"Metformax, galvus, amaryl",1



Podsumowanie nowej kolumny dummy:
ank_zdro_leki_cukrzyca_D
0    206
1      5
Name: count, dtype: int64


```markdown
### Przegląd wszystkich leków w kolumnie `ank_zdro_leki`  
Poniższy kod wyciągnie unikalne wpisy, aby sprawdzić, czy nie pominęliśmy żadnych leków na cukrzycę lub innych istotnych substancji.
```

In [24]:
# Wyciągnięcie unikalnych wpisów leków, posortowanych alfabetycznie
all_meds = df['ank_zdro_leki'].dropna().unique().tolist()
all_meds_cleaned = sorted(list(set([str(m).strip() for m in all_meds])))

print(f"Liczba unikalnych wpisów: {len(all_meds_cleaned)}")
print("--- Lista wszystkich wpisów w 'ank_zdro_leki' ---")
for med in all_meds_cleaned:
    print(med)

Liczba unikalnych wpisów: 77
--- Lista wszystkich wpisów w 'ank_zdro_leki' ---
Acar, zahron duo, co-prestarium, bisovation, dalfor uno, polomag B6, nefrocal, prostaceum
Adatam, lipanthyl odstawiony rok temu
Amlessa
Amlozeh, zahron
Atrox
BEtaloc, Flixonase
BRAK
Betalol, Tritace
Betaserc
Betozic, Trutace, Jovesto, Acard, Melatonina 5G
Bisocard, Valsacor, Omnilocas
Candezek Combi 8/5
Carzap, hitaxa
Cetryzyna okresowo
Co-Amlessa (4+5+1,25); Kalipoz; Neomag skurcz; Prostamol uno
Co-Amlessa 8mg+5mg+2,5mg
Co-prenesse, esseliv forte
Co-prestarium 10 mg+10 mg
Co-prestarium 5mg+5 mg, contrahist
Comlessa 8+10+250; Nobilet; SioFou 850
Concor, tertensif
Controloc 20
Dipperam 5 mg + 70mg
Elicea 10 mg
Euthyrox 88
Ezehron Duo, polprazol pph, polpril, tritace 5  Ezehron Duo, polprazol pph, polpril, tritace 5
Flonidan, aerozole do nosa, witaminy D3, C, rutyna, magnez
Fostex, telmizek
Glucophag, Prestarium 5mg. Romazic, Milurit 100mg
Indapen, trittico
Indix combi, nebilet, amlopin, atoris
Klatra, monkast

In [27]:
df.columns.tolist()

['Unnamed: 0',
 'id',
 'hig_sub_oc_zebow',
 'hig_sub_oc_zebow_dych',
 'hig_sub_oc_dziasel',
 'hig_sub_oc_dziasel_dych',
 'hig_kiedy_myje_zeby',
 'hig_myje_zeby_od_razu_po_przebudzeniu',
 'hig_myje_zeby_przed_polożeniem_się_spać',
 'hig_myje_zeby_po_każdym_dużym_posilku',
 'hig_myje_zeby_przed_wyjściem_z_domu',
 'hig_myje_zeby_po_każdej_przekąsce',
 'hig_szczotka',
 'hig_szczotka_U',
 'hig_plyn_plukanie',
 'hig_plyn_plukanie_U',
 'hig_nic_dent',
 'hig_nic_dent_U',
 'hig_mycia_zebow',
 'hig_mycia_zebow_U',
 'hig_mycia_zebow_zmiana',
 'hig_mycia_zebow_zmiana_kiedy',
 'hig_mycia_zebow_przeszlosc',
 'hig_wiz_stoma_przeg',
 'hig_wiz_stoma_przeg_zmiana',
 'hig_wiz_stoma_przeg_zmiana_kiedy',
 'hig_wiz_stoma_przeg_zmiana_przeszlosc',
 'hig_wiz_stoma_inter',
 'hig_wiz_stoma_inter_zmiana',
 'hig_wiz_stoma_inter_zmiana_kiedy',
 'hig_wiz_stoma_inter_zmiana_przeszlosc',
 'hig_jedz_slodycz_teraz',
 'hig_jedz_slodkie_napoje_teraz',
 'hig_jedz_owoce_teraz',
 'hig_jedz_slodka_kawa_teraz',
 'hig_jedz_slo

In [28]:
df[]


,Unnamed: 0,id,hig_sub_oc_zebow,hig_sub_oc_zebow_dych,hig_sub_oc_dziasel,hig_sub_oc_dziasel_dych,hig_kiedy_myje_zeby,hig_myje_zeby_od_razu_po_przebudzeniu,hig_myje_zeby_przed_polożeniem_się_spać,hig_myje_zeby_po_każdym_dużym_posilku,...,pom_pal_l_5,hsCRP,sIgA,ank_zdro_alko_U,ank_zdro_leki_U,ank_zdro_chor_przew_U,ank_zdro_chor_przew_cukrz,ank_zdro_leki_cukrzyca_D,par_hg_glikowana_norma,par_ob_norma
0,0,1,3.0,NizszaOcena,4.0,NizszaOcena,"Od razu po przebudzeniu,Po każdym dużym posiłk...",1.0,1.0,1.0,...,63.110,NaN,NaN,4 razy w tygodniu i częściej,NIE,Brak,Brak,0,1.0,1.0
1,1,2,4.0,NizszaOcena,6.0,WyzszaOcena,"Od razu po przebudzeniu,Przed położeniem się spać",1.0,1.0,0.0,...,67.585,NaN,NaN,nigdy,NIE,Brak,Brak,0,1.0,1.0
2,2,3,5.0,WyzszaOcena,5.0,WyzszaOcena,"Przed wyjściem z domu,Przed położeniem się spać",0.0,1.0,0.0,...,65.110,NaN,NaN,raz w miesiącu lub rzadziej,NIE,Brak,Brak,0,1.0,1.0
3,3,4,5.0,WyzszaOcena,4.0,NizszaOcena,"Od razu po przebudzeniu,Przed położeniem się spać",1.0,1.0,0.0,...,59.060,NaN,NaN,raz w miesiącu lub rzadziej,NIE,Brak,Brak,0,1.0,1.0
4,4,5,5.0,WyzszaOcena,5.0,WyzszaOcena,"Od razu po przebudzeniu,Po każdym dużym posiłk...",1.0,1.0,1.0,...,61.945,0.829224,2032.905,raz w miesiącu lub rzadziej,TAK,Brak,Brak,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,206,207,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,63.570,0.400810,319.200,nigdy,TAK,Brak,Brak,0,1.0,1.0
207,207,208,5.0,WyzszaOcena,4.0,NizszaOcena,"Przed wyjściem z domu,Przed położeniem się spać",0.0,1.0,0.0,...,60.245,0.732794,4136.990,raz w miesiącu lub rzadziej,NIE,Brak,Brak,0,1.0,1.0
208,208,209,4.0,NizszaOcena,3.0,NizszaOcena,Przed położeniem się spać,0.0,1.0,0.0,...,62.605,0.465587,NaN,raz w miesiącu lub rzadziej,NIE,Brak,Brak,0,1.0,1.0
209,209,210,4.0,NizszaOcena,4.0,NizszaOcena,Po każdym dużym posiłku,0.0,0.0,1.0,...,64.595,1.080972,862.020,raz w miesiącu lub rzadziej,NIE,Refluks/GERD,Refluks (od 20 lat),0,1.0,1.0


In [31]:
# Tworzenie kolumny dummy dla obecności oceny zębów
df['hig_sub_oc_zebow_dummy'] = df['hig_sub_oc_zebow'].notna().astype(int)

# Sprawdzenie wyników
print("Podsumowanie obecności danych w 'hig_sub_oc_zebow':")
print(df['hig_sub_oc_zebow_dummy'].value_counts())
display(df[['hig_sub_oc_zebow', 'hig_sub_oc_zebow_dummy']].head(10))

Podsumowanie obecności danych w 'hig_sub_oc_zebow':
hig_sub_oc_zebow_dummy
1    154
0     57
Name: count, dtype: int64


,hig_sub_oc_zebow,hig_sub_oc_zebow_dummy
0,3.0,1
1,4.0,1
2,5.0,1
3,5.0,1
4,5.0,1
5,NaN,0
6,5.0,1
7,NaN,0
8,4.0,1
9,NaN,0
